In [1]:
import os
import glob
import numpy as np
from nilearn.image import concat_imgs, load_img, new_img_like, smooth_img
from scipy.stats import ttest_1samp
from nilearn.plotting import plot_stat_map

In [2]:
# ---- Settings ----
searchlight_dir = "/Users/liuq13/SPCD_data/searchlight_maps/odor_scan"
output_dir = os.path.join(searchlight_dir, "group_stats")
os.makedirs(output_dir, exist_ok=True)
chance_level = 0.5
fwhm = 6  # smoothing kernel in mm

In [3]:
# ---- Step 1: Load and smooth individual maps ----
map_files = sorted(glob.glob(os.path.join(searchlight_dir, "sub-*_searchlight.nii.gz")))
print(f"Found {len(map_files)} subject maps")

smoothed_maps = []
for f in map_files:
    smoothed = smooth_img(f, fwhm=fwhm)
    smoothed_maps.append(smoothed)

    # Optional: save smoothed image for reference
    sub_id = os.path.basename(f).split('_')[0]  # e.g., 'sub-01'
    smoothed_path = os.path.join(searchlight_dir, f"{sub_id}_smoothed.nii.gz")
    smoothed.to_filename(smoothed_path)

Found 34 subject maps


In [4]:
# ---- Step 2: Concatenate into 4D image ----
img_4d = concat_imgs(smoothed_maps)
data_4d = img_4d.get_fdata()  # shape: (X, Y, Z, N)
data_4d.shape

(79, 95, 79, 34)

In [5]:
# ---- Step 3: Reshape and run t-test ----
n_voxels = np.prod(data_4d.shape[:-1])
n_subjects = data_4d.shape[-1]
data_2d = data_4d.reshape(n_voxels, n_subjects)
data_2d.shape

(592895, 34)

In [6]:
# Mask out any voxel with NaNs across subjects
valid_mask = ~np.any(np.isnan(data_2d), axis=1)
valid_data = data_2d[valid_mask, :]
valid_data.shape

(592895, 34)

In [7]:
# ---- Step 4: One-sample t-test vs. chance ----
t_vals, p_vals = ttest_1samp(valid_data, popmean=chance_level, axis=1)

# ---- Step 5: Reconstruct t-map into 3D ----
t_map_flat = np.full(n_voxels, np.nan)
t_map_flat[valid_mask] = t_vals
t_map_3d = t_map_flat.reshape(data_4d.shape[:-1])

# ---- Step 6: Save t-map ----
t_map_img = new_img_like(img_4d.slicer[..., 0], t_map_3d)
t_map_path = os.path.join(output_dir, f"group_tmap_smoothed_fwhm{fwhm}.nii.gz")
t_map_img.to_filename(t_map_path)
print(f"Saved t-map to: {t_map_path}")

Saved t-map to: /Users/liuq13/SPCD_data/searchlight_maps/odor_scan/group_stats/group_tmap_smoothed_fwhm6.nii.gz
